# ISIC 2024 - Patient Folds and Benchmarks

1. Drop the leaky columns
2. Make 5 patient-based folds (StratifiedGroupKFold)
3. Check the folds
4. Benchmark 1: `-tbp_lv_H` (no training)
5. Save the folds so every later model uses the same split
6. Benchmark 2: logistic regression on the metadata

In [10]:
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

sys.path.insert(0, '../src')
from isic_pauc import p_auc_tpr   # official ISIC 2024 metric

DATA_DIR = '../data'
SEED = 42
N_FOLDS = 5

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train-metadata.csv', low_memory=False)
test = pd.read_csv(f'{DATA_DIR}/test-metadata.csv')
train.shape

(401059, 55)

## 1. Drop leaky columns
Columns that are in train but not in test (except `target`).

In [4]:
leak_cols = [col for col in train.columns if col not in test.columns and col != 'target']
print(leak_cols)

train = train.drop(columns=leak_cols)
train.shape

['lesion_id', 'iddx_full', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5', 'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence']


(401059, 45)

## 2. Patient-based folds
`StratifiedGroupKFold`: every patient stays in one fold (group), and each fold gets a similar share of malignant lesions (stratified).

In [5]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

train['fold'] = -1
X = train.drop(columns=['target'])
y = train['target']
groups = train['patient_id']
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X=X, y=y, groups=groups)):
    train.loc[val_idx, 'fold'] = fold

train['fold'].value_counts().sort_index()

fold
0    80211
1    80211
2    80216
3    80211
4    80210
Name: count, dtype: int64

## 3. Check the folds
Each fold should have a similar number of malignant lesions, and no patient should be in more than one fold.

In [6]:
fold_summary = train.groupby('fold').agg(
    rows=('target', 'size'),
    malignant_lesions=('target', 'sum'),
    malignant_rate=('target', 'mean'),
    patients=('patient_id', 'nunique')
)

fold_summary

,rows,malignant_lesions,malignant_rate,patients
fold,,,,
0,80211,78,0.000972,207
1,80211,78,0.000972,208
2,80216,83,0.001035,210
3,80211,77,0.000960,209
4,80210,77,0.000960,208


In [7]:
patient_folds = train.groupby('patient_id')['fold'].nunique()

assert patient_folds.max() == 1, 'a patient is in more than one fold!'

## 4. Benchmark 1: `-tbp_lv_H`
No training, just rank the lesions by one feature.

- Lower H = more likely malignant, so the score has to be flipped (higher score = more likely malignant).
- `p_auc_tpr` expects scores between 0 and 1. `rankdata(x) / len(x)` keeps the order and puts everything in (0, 1].

In [18]:
paucs, aucs = [], []

for fold in range(N_FOLDS):
    val = train[train['fold'] == fold]

    score = rankdata(-val['tbp_lv_H']) / len(val['tbp_lv_H'])

    pauc = p_auc_tpr(val['target'], score, 0.80)
    auc = roc_auc_score(val['target'], score)

    paucs.append(pauc)
    aucs.append(auc)
    print(f'fold {fold}: pAUC={pauc:.4f}  AUC={auc:.4f}')

print(f'Benchmark -tbp_lv_H  pAUC: {np.mean(paucs):.4f} +/- {np.std(paucs):.4f}')
print(f'Benchmark -tbp_lv_H  AUC : {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}')

fold 0: pAUC=0.0902  AUC=0.8371
fold 1: pAUC=0.0820  AUC=0.8040
fold 2: pAUC=0.1197  AUC=0.8698
fold 3: pAUC=0.0451  AUC=0.7265
fold 4: pAUC=0.0676  AUC=0.7802
Benchmark -tbp_lv_H  pAUC: 0.0809 +/- 0.0247
Benchmark -tbp_lv_H  AUC : 0.8035 +/- 0.0490


## 5. Save the folds
Saving them so every model after this uses exactly the same split.

In [19]:
train[['isic_id', 'patient_id', 'target', 'fold']].to_csv(
    '../data/folds.csv',
    index=False
)
pd.read_csv('../data/folds.csv').head()

,isic_id,patient_id,target,fold
0,ISIC_0015670,IP_1235828,0,1
1,ISIC_0015845,IP_8170065,0,4
2,ISIC_0015864,IP_6724798,0,4
3,ISIC_0015902,IP_4111386,0,1
4,ISIC_0024200,IP_8313778,0,1


## 6. Benchmark 2: Logistic regression on the metadata
- Only the numeric non-leak columns (no ids, `target` or `fold`).
- Mean imputation -> `StandardScaler` -> `LogisticRegression` in one pipeline.
- The pipeline is fitted only on the training folds, so the mean and scaling don't leak from the validation fold.

In [28]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

feature_cols = train.select_dtypes('number').drop(columns=['target', 'fold']).columns
print(len(feature_cols))
feature_cols

34


Index(['age_approx', 'clin_size_long_diam_mm', 'tbp_lv_A', 'tbp_lv_Aext',
       'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H',
       'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2',
       'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA',
       'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB',
       'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_minorAxisMM',
       'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color',
       'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL',
       'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle',
       'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z'],
      dtype='str')

In [29]:
lr_paucs, lr_aucs = [], []

for fold in range(N_FOLDS):
    trn = train[train['fold'] != fold]
    val = train[train['fold'] == fold]

    model = make_pipeline(
        SimpleImputer(strategy='mean'),
        StandardScaler(),
        LogisticRegression(max_iter=1000),
    )

    model.fit(trn[feature_cols], trn['target'])

    pred = model.predict_proba(val[feature_cols])[:, 1]

    pauc = p_auc_tpr(val['target'], pred, 0.80)
    auc = roc_auc_score(val['target'], pred)

    lr_paucs.append(pauc)
    lr_aucs.append(auc)
    print(f'fold {fold}: pAUC={pauc:.4f}  AUC={auc:.4f}')

print(f'Logistic regression  pAUC: {np.mean(lr_paucs):.4f} +/- {np.std(lr_paucs):.4f}')
print(f'Logistic regression  AUC : {np.mean(lr_aucs):.4f} +/- {np.std(lr_aucs):.4f}')

fold 0: pAUC=0.1346  AUC=0.9086
fold 1: pAUC=0.1054  AUC=0.8665
fold 2: pAUC=0.1363  AUC=0.9042
fold 3: pAUC=0.0855  AUC=0.8096
fold 4: pAUC=0.0964  AUC=0.8529
Logistic regression  pAUC: 0.1116 +/- 0.0205
Logistic regression  AUC : 0.8684 +/- 0.0363


In [30]:
# Compare both benchmarks per fold
pd.DataFrame({
    'H_pAUC': paucs,
    'LR_pAUC': lr_paucs,
    'H_AUC': aucs,
    'LR_AUC': lr_aucs,
}, index=[f'fold {i}' for i in range(N_FOLDS)])

,H_pAUC,LR_pAUC,H_AUC,LR_AUC
fold 0,0.090161,0.134630,0.837094,0.908596
fold 1,0.082018,0.105357,0.803958,0.866520
fold 2,0.119718,0.136336,0.869802,0.904181
fold 3,0.045144,0.085503,0.726499,0.809647
fold 4,0.067614,0.096403,0.780152,0.852874
